# Dantzig, Fulkerson, and Johnson's Large-Scale Traveling-Salesman Problem

In 1954, George Dantzig, Ray Fulkerson, and Selmer Johnson solved a road-distance traveling-salesman problem with one city in each of the 48 states then in the Union plus Washington, D.C. Their paper introduced the cutting-plane idea now known as the **subtour-elimination formulation** of the TSP.

The actual optimization used 42 numbered cities. Seven cities labeled A through G lie, in order, on the shortest road from city 40 (Washington) to city 41 (Boston), so inserting them turns the optimal 42-city cycle into the claimed 49-city tour without changing that optimization.

For an undirected edge variable $x_{ij}$, the paper begins with degree equations

$$\sum_{j\ne i}x_{ij}=2 \qquad (i=1,\ldots,n),$$

then excludes every disconnected loop on a proper city subset $S$ with

$$\sum_{i\in S,\;j\notin S}x_{ij}\ge 2.$$

The published optimum is **699 adjusted distance units**, reported as **12,345 road miles** before the paper's scaling and rounding.

## The 42-City Computational Instance

The cell below embeds the canonical TSPLIB transcription of the paper's lower-triangular distance table and its plotting coordinates. This keeps the notebook reproducible without a network request.

In [ ]:
%%capture
! pip install rhetor matplotlib bokeh bokeh_sampledata

In [1]:
import numpy as np

CITY_NAMES = [
    "Manchester, NH", "Montpelier, VT", "Detroit, MI", "Cleveland, OH",
    "Charleston, WV", "Louisville, KY", "Indianapolis, IN", "Chicago, IL",
    "Milwaukee, WI", "Minneapolis, MN", "Pierre, SD", "Bismarck, ND",
    "Helena, MT", "Seattle, WA", "Portland, OR", "Boise, ID",
    "Salt Lake City, UT", "Carson City, NV", "Los Angeles, CA", "Phoenix, AZ",
    "Santa Fe, NM", "Denver, CO", "Cheyenne, WY", "Omaha, NE",
    "Des Moines, IA", "Kansas City, MO", "Topeka, KS", "Oklahoma City, OK",
    "Dallas, TX", "Little Rock, AR", "Memphis, TN", "Jackson, MS",
    "New Orleans, LA", "Birmingham, AL", "Atlanta, GA", "Jacksonville, FL",
    "Columbia, SC", "Raleigh, NC", "Richmond, VA", "Washington, DC",
    "Boston, MA", "Portland, ME",
]

LOWER_DIAG_ROW = """
0 8 0 39 45 0 37 47 9 0 50 49 21 15 0 61 62 21 20 17 0 58 60 16 17 18 6 0
59 60 15 20 26 17 10 0 62 66 20 25 31 22 15 5 0 81 81 40 44 50 41 35 24 20 0
103 107 62 67 72 63 57 46 41 23 0 108 117 66 71 77 68 61 51 46 26 11 0
145 149 104 108 114 106 99 88 84 63 49 40 0 181 185 140 144 150 142 135 124 120 99 85 76 35 0
187 191 146 150 156 142 137 130 125 105 90 81 41 10 0
161 170 120 124 130 115 110 104 105 90 72 62 34 31 27 0
142 146 101 104 111 97 91 85 86 75 51 59 29 53 48 21 0
174 178 133 138 143 129 123 117 118 107 83 84 54 46 35 26 31 0
185 186 142 143 140 130 126 124 128 118 93 101 72 69 58 58 43 26 0
164 165 120 123 124 106 106 105 110 104 86 97 71 93 82 62 42 45 22 0
137 139 94 96 94 80 78 77 84 77 56 64 65 90 87 58 36 68 50 30 0
117 122 77 80 83 68 62 60 61 50 34 42 49 82 77 60 30 62 70 49 21 0
114 118 73 78 84 69 63 57 59 48 28 36 43 77 72 45 27 59 69 55 27 5 0
85 89 44 48 53 41 34 28 29 22 23 35 69 105 102 74 56 88 99 81 54 32 29 0
77 80 36 40 46 34 27 19 21 14 29 40 77 114 111 84 64 96 107 87 60 40 37 8 0
87 89 44 46 46 30 28 29 32 27 36 47 78 116 112 84 66 98 95 75 47 36 39 12 11 0
91 93 48 50 48 34 32 33 36 30 34 45 77 115 110 83 63 97 91 72 44 32 36 9 15 3 0
105 106 62 63 64 47 46 49 54 48 46 59 85 119 115 88 66 98 79 59 31 36 42 28 33 21 20 0
111 113 69 71 66 51 53 56 61 57 59 71 96 130 126 98 75 98 85 62 38 47 53 39 42 29 30 12 0
91 92 50 51 46 30 34 38 43 49 60 71 103 141 136 109 90 115 99 81 53 61 62 36 34 24 28 20 20 0
83 85 42 43 38 22 26 32 36 51 63 75 106 142 140 112 93 126 108 88 60 64 66 39 36 27 31 28 28 8 0
89 91 55 55 50 34 39 44 49 63 76 87 120 155 150 123 100 123 109 86 62 71 78 52 49 39 44 35 24 15 12 0
95 97 64 63 56 42 49 56 60 75 86 97 126 160 155 128 104 128 113 90 67 76 82 62 59 49 53 40 29 25 23 11 0
74 81 44 43 35 23 30 39 44 62 78 89 121 159 155 127 108 136 124 101 75 79 81 54 50 42 46 43 39 23 14 14 21 0
67 69 42 41 31 25 32 41 46 64 83 90 130 164 160 133 114 146 134 111 85 84 86 59 52 47 51 53 49 32 24 24 30 9 0
74 76 61 60 42 44 51 60 66 83 102 110 147 185 179 155 133 159 146 122 98 105 107 79 71 66 70 70 60 48 40 36 33 25 18 0
57 59 46 41 25 30 36 47 52 71 93 98 136 172 172 148 126 158 147 124 121 97 99 71 65 59 63 67 62 46 38 37 43 23 13 17 0
45 46 41 34 20 34 38 48 53 73 96 99 137 176 178 151 131 163 159 135 108 102 103 73 67 64 69 75 72 54 46 49 54 34 24 29 12 0
35 37 35 26 18 34 36 46 51 70 93 97 134 171 176 151 129 161 163 139 118 102 101 71 65 65 70 84 78 58 50 56 62 41 32 38 21 9 0
29 33 30 21 18 35 33 40 45 65 87 91 117 166 171 144 125 157 156 139 113 95 97 67 60 62 67 79 82 62 53 59 66 45 38 45 27 15 6 0
3 11 41 37 47 57 55 58 63 83 105 109 147 186 188 164 144 176 182 161 134 119 116 86 78 84 88 101 108 88 80 86 92 71 64 71 54 41 32 25 0
5 12 55 41 53 64 61 61 66 84 111 113 150 186 192 166 147 180 188 167 140 124 119 90 87 90 94 107 114 77 86 92 98 80 74 77 60 48 38 32 6 0
"""

COORDINATES = np.array([
    (170, 85), (166, 88), (133, 73), (140, 70), (142, 55), (126, 53), (125, 60),
    (119, 68), (117, 74), (99, 83), (73, 79), (72, 91), (37, 94), (6, 106),
    (3, 97), (21, 82), (33, 67), (4, 66), (3, 42), (27, 33), (52, 41),
    (57, 59), (58, 66), (88, 65), (99, 67), (95, 55), (89, 55), (83, 38),
    (85, 25), (104, 35), (112, 37), (112, 24), (113, 13), (125, 30),
    (135, 32), (147, 18), (147.5, 36), (154.5, 45), (157, 54), (158, 61),
    (172, 82), (174, 87),
], dtype=float)

values = [int(value) for value in LOWER_DIAG_ROW.split()]
assert len(values) == 42 * 43 // 2
DISTANCE = np.zeros((42, 42), dtype=int)
position = 0
for row in range(42):
    for column in range(row + 1):
        DISTANCE[row, column] = DISTANCE[column, row] = values[position]
        position += 1

PUBLISHED_TOUR = list(range(42)) + [0]
published_length = sum(DISTANCE[i, j] for i, j in zip(PUBLISHED_TOUR, PUBLISHED_TOUR[1:]))

assert len(CITY_NAMES) == len(COORDINATES) == DISTANCE.shape[0] == 42
assert np.array_equal(DISTANCE, DISTANCE.T)
assert np.all(np.diag(DISTANCE) == 0)
assert published_length == 699

print(f"Cities: {len(CITY_NAMES)}")
print(f"Symmetric distance entries: {len(values)}")
print(f"Published cycle length: {published_length} adjusted units")

Cities: 42
Symmetric distance entries: 903
Published cycle length: 699 adjusted units


## Reproducing the Cutting-Plane Method

Dantzig, Fulkerson, and Johnson first minimized distance subject to the degree equations. A binary degree-feasible solution can contain several disjoint cycles. Whenever that happens, each component $S$ supplies a violated loop condition

$$\sum_{i\in S,\;j\notin S}x_{ij}\ge 2.$$

This constraint requires that any non-trivial subset $S$ must be connected to the rest of the graph by at least 2 edges to ensure a single continuous loop.

The paper performed simplex pivots and cut selection by hand, allowed fractional intermediate solutions, and ultimately used two additional instance-specific inequalities in its LP certificate. The reconstruction below keeps the paper's decisive algorithmic idea while solving a binary master problem at each iteration. Thus every intermediate solution is a collection of cycles, and ordinary loop conditions suffice. This is the modern integer cutting-plane version of the original procedure, rather than a line-for-line simulation of its hand calculations.

In [2]:
import itertools

import networkx as nx
import pandas as pd
from scipy.optimize import Bounds, LinearConstraint, milp
from scipy.sparse import lil_matrix

EDGES = list(itertools.combinations(range(42), 2))
EDGE_INDEX = {edge: index for index, edge in enumerate(EDGES)}
EDGE_COST = np.array([DISTANCE[i, j] for i, j in EDGES], dtype=float)


def solve_degree_model(cuts):
    rows = 42 + len(cuts)
    matrix = lil_matrix((rows, len(EDGES)), dtype=float)
    lower = np.full(rows, 2.0)
    upper = np.full(rows, 2.0)

    for city in range(42):
        for edge, index in EDGE_INDEX.items():
            if city in edge:
                matrix[city, index] = 1.0

    for cut_number, subset in enumerate(cuts, start=42):
        for (i, j), index in EDGE_INDEX.items():
            if (i in subset) != (j in subset):
                matrix[cut_number, index] = 1.0
        upper[cut_number] = np.inf

    result = milp(
        c=EDGE_COST,
        integrality=np.ones(len(EDGES)),
        bounds=Bounds(0, 1),
        constraints=LinearConstraint(matrix.tocsr(), lower, upper),
        options={"disp": False},
    )
    assert result.success
    selected = [EDGES[index] for index, value in enumerate(result.x) if value > 0.5]
    graph = nx.Graph()
    graph.add_nodes_from(range(42))
    graph.add_edges_from(selected)
    components = [frozenset(component) for component in nx.connected_components(graph)]
    return round(result.fun), selected, components


cuts = []
cut_keys = set()
history = []
while True:
    objective, historical_edges, components = solve_degree_model(cuts)
    history.append(
        {
            "iteration": len(history) + 1,
            "lower_bound": objective,
            "components": len(components),
            "component_sizes": sorted(map(len, components)),
            "cuts_in_model": len(cuts),
        }
    )
    if len(components) == 1:
        break
    for component in components:
        canonical = component if len(component) <= 21 else frozenset(set(range(42)) - component)
        if canonical not in cut_keys:
            cut_keys.add(canonical)
            cuts.append(canonical)

history_frame = pd.DataFrame(history)
display(history_frame)
assert objective == 699
assert len(historical_edges) == 42
print(f"Connected tour found after {len(history)} solves and {len(cuts)} loop cuts.")

,iteration,lower_bound,components,component_sizes,cuts_in_model
0,1,646,8,"[3, 3, 4, 4, 4, 6, 7, 11]",0
1,2,688,2,"[13, 29]",8
2,3,699,1,[42],9


Connected tour found after 3 solves and 9 loop cuts.


## Exact TSP Formulation in PyOPL

For an independent exact solve, the PyOPL model uses directed binary variables $x_{ij}$. Every city has one entering and one leaving arc. Continuous order variables $u_i$ satisfy the Miller-Tucker-Zemlin inequalities

$$u_j\ge u_i+1-n(1-x_{ij}) \qquad (i,j\ne 1,\ i\ne j),$$

which forbid a cycle that omits the fixed root city. This compact formulation differs from the paper's dynamically generated undirected cuts, but represents the same TSP and provides a useful independent verification.

In [2]:
model_text = r'''
/*
 * Dantzig-Fulkerson-Johnson Traveling-Salesman Problem
 * ---------------------------------------------------
 * In 1954, George Dantzig, Ray Fulkerson, and Selmer Johnson solved a
 * road-distance TSP covering the 48 states then in the Union and
 * Washington, D.C. Their paper introduced the cutting-plane approach
 * now associated with subtour elimination.
 *
 * The optimization used 42 numbered cities. Seven additional cities,
 * labeled A through G, lie in order on the shortest road from city 40
 * (Washington) to city 41 (Boston). Inserting them converts the optimal
 * 42-city cycle into the reported 49-city tour without changing the
 * underlying optimization.
 *
 * Reference:
 * G. B. Dantzig, R. Fulkerson, and S. Johnson,
 * "Solution of a Large-Scale Traveling-Salesman Problem,"
 * Journal of the Operations Research Society of America,
 * 2(4), 393-410, 1954.
 */

// § Parameters

// Number of cities and their index range.
int N = ...;
range Cities = 1..N;

// Road distance from city i to city j.
param int distance[Cities][Cities];

// § Decision variables

// x[i][j] = 1 iff the tour travels directly from city i to city j.
dvar boolean x[Cities][Cities];

// Continuous MTZ order assigned to each city, bounded by 1 and N.
dvar float+ order[Cities] in 1..N;

// § Objective

// Minimize the total distance of the selected tour arcs.
minimize TotalAdjustedDistance:
  sum(i in Cities, j in Cities: i != j) distance[i][j] * x[i][j];

subject to {
  // § Arc restrictions

  // Exclude self-loops.
  forall(i in Cities)
    noSelfLoop: x[i][i] == 0;

  // § Degree constraints

  // Each city has exactly one outgoing arc.
  forall(i in Cities)
    oneDeparture: sum(j in Cities: j != i) x[i][j] == 1;

  // Each city has exactly one incoming arc.
  forall(j in Cities)
    oneArrival: sum(i in Cities: i != j) x[i][j] == 1;

  // § MTZ subtour elimination

  // Anchor the order numbering at city 1.
  rootOrder: order[1] == 1;

  // Reserve order 1 for the root city.
  forall(i in 2..N)
    orderLowerBound: order[i] >= 2;

  // If arc (i,j) is selected between non-root cities, j must follow i.
  // These ordering constraints eliminate subtours that exclude city 1.
  forall(i in 2..N, j in 2..N: i != j)
    subtourElimination:
      order[j] >= order[i] + 1 - N * (1 - x[i][j]);
}
'''

matrix_rows = ["  [" + ", ".join(map(str, row)) + "]" for row in DISTANCE]
data_text = "N = 42;\ndistance = [\n" + ",\n".join(matrix_rows) + "\n];\n"

print(model_text)
print(f"Data contains a {DISTANCE.shape[0]} x {DISTANCE.shape[1]} distance matrix.")


/*
 * Dantzig-Fulkerson-Johnson Traveling-Salesman Problem
 * ---------------------------------------------------
 * In 1954, George Dantzig, Ray Fulkerson, and Selmer Johnson solved a
 * road-distance TSP covering the 48 states then in the Union and
 * Washington, D.C. Their paper introduced the cutting-plane approach
 * now associated with subtour elimination.
 *
 * The optimization used 42 numbered cities. Seven additional cities,
 * labeled A through G, lie in order on the shortest road from city 40
 * (Washington) to city 41 (Boston). Inserting them converts the optimal
 * 42-city cycle into the reported 49-city tour without changing the
 * underlying optimization.
 *
 * Reference:
 * G. B. Dantzig, R. Fulkerson, and S. Johnson,
 * "Solution of a Large-Scale Traveling-Salesman Problem,"
 * Journal of the Operations Research Society of America,
 * 2(4), 393-410, 1954.
 */

// § Parameters

// Number of cities and their index range.
int N = ...;
range Cities = 1..N;

// Road distan

## Solving with PyOPL and Gurobi

PyOPL compiles the OPL model and delegates the mixed-integer optimization to Gurobi. The repository-local environment already contains PyOPL and `gurobipy`; executing the next cell also requires a valid Gurobi license.

In [4]:
from pathlib import Path

from pyopl import solve

model_file = Path("dantzig42.mod")
data_file = Path("dantzig42.dat")
model_file.write_text(model_text)
data_file.write_text(data_text)

results = solve(str(model_file), str(data_file), solver="gurobi")
stats = results.get("stats", {})

print(f"Status: {results['status']}")
print(f"Objective: {results['objective_value']:.0f} adjusted units")
print(f"MIP gap: {stats.get('MIPGap', float('nan')):.2%}")
print(f"Runtime: {stats.get('Runtime', float('nan')):.2f} seconds")

assert results["status"] == "OPTIMAL"
assert abs(results["objective_value"] - 699) < 1e-6

PyOPL: compiling model...
Set parameter Username
Set parameter LicenseID to value 2780509
Academic license - for non-commercial use only - expires 2027-02-18
Set parameter OutputFlag to value 1
Set parameter LogToConsole to value 1
PyOPL/Gurobi: variables=1806, constraints=0, sense=1
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 1849 rows, 1806 columns and 8489 nonzeros (Min)
Model fingerprint: 0x88affc27
Model has 1722 linear objective coefficients
Variable types: 42 continuous, 1764 integer (1764 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+01]
  Objective range  [3e+00, 2e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 4e+01]

Presolve removed 125 rows and 43 columns
Presolve time: 0.03s
Presolved: 1724 rows, 1763 columns, 8364 nonzeros
Variable types: 41 continuous, 1722 integer (1722 bina

## Extract and Validate the Optimal Tour

The PyOPL solution is directed, so the selected arcs define a successor for each city. Starting at city 1 and following those successors must return to city 1 after exactly 42 moves. Since this symmetric TSP may be traversed in either direction, the validation compares edge sets and objective values rather than requiring one orientation.

In [5]:
import re

arc_pattern = re.compile(r"x(?:\[(\d+),(\d+)\]|_(\d+)_(\d+))")


def parse_arc(variable_name):
    match = arc_pattern.fullmatch(variable_name)
    if not match:
        return None
    indices = match.group(1, 2) if match.group(1) is not None else match.group(3, 4)
    return tuple(int(index) - 1 for index in indices)


selected_arcs = []
for variable_name, value in results["solution"].items():
    arc = parse_arc(variable_name)
    if arc is not None and value > 0.5:
        selected_arcs.append(arc)

successor = dict(selected_arcs)
optimal_tour = [0]
for _ in range(42):
    optimal_tour.append(successor[optimal_tour[-1]])

recomputed_length = sum(DISTANCE[i, j] for i, j in zip(optimal_tour, optimal_tour[1:]))
pyopl_edges = {frozenset(edge) for edge in selected_arcs}
historical_edge_set = {frozenset(edge) for edge in historical_edges}
published_edges = {frozenset(edge) for edge in zip(PUBLISHED_TOUR, PUBLISHED_TOUR[1:])}

assert len(selected_arcs) == 42
assert optimal_tour[0] == optimal_tour[-1] == 0
assert sorted(optimal_tour[:-1]) == list(range(42))
assert recomputed_length == results["objective_value"] == objective == published_length == 699
assert pyopl_edges == historical_edge_set == published_edges

print(" -> ".join(str(city + 1) for city in optimal_tour))
print(f"Validated cycle length: {recomputed_length} adjusted units")
print("The Gurobi tour is the published cycle in reverse orientation.")

1 -> 42 -> 41 -> 40 -> 39 -> 38 -> 37 -> 36 -> 35 -> 34 -> 33 -> 32 -> 31 -> 30 -> 29 -> 28 -> 27 -> 26 -> 25 -> 24 -> 23 -> 22 -> 21 -> 20 -> 19 -> 18 -> 17 -> 16 -> 15 -> 14 -> 13 -> 12 -> 11 -> 10 -> 9 -> 8 -> 7 -> 6 -> 5 -> 4 -> 3 -> 2 -> 1
Validated cycle length: 699 adjusted units
The Gurobi tour is the published cycle in reverse orientation.


## Display the Optimal Tour on a U.S. Map

The map uses approximate geographic coordinates for the 42 cities and applies the same Albers equal-area projection to both the cities and the contiguous-state polygons from `bokeh_sampledata`. The projected straight segments indicate tour order; the optimization still uses the paper's road-distance matrix rather than geometric distance. Install `bokeh` and `bokeh_sampledata` if those packages are unavailable.

Numbered markers correspond to the paper's 42-city optimization. The seven additional stops are listed separately because the paper inserts them along the shortest Washington-to-Boston road segment rather than optimizing them as independent nodes.

| Paper label | Inserted city |
|---|---|
| A | Baltimore, MD |
| B | Wilmington, DE |
| C | Philadelphia, PA |
| D | Newark, NJ |
| E | New York, NY |
| F | Hartford, CT |
| G | Providence, RI |

In [6]:
from io import StringIO

import matplotlib.pyplot as plt
from bokeh.sampledata.us_states import data as US_STATES
from IPython.display import HTML, display
from matplotlib.patches import FancyArrowPatch

CITY_LON_LAT = np.array([
    (-71.4548, 42.9956), (-72.5754, 44.2601), (-83.0458, 42.3314), (-81.6944, 41.4993),
    (-81.6326, 38.3498), (-85.7585, 38.2527), (-86.1581, 39.7684), (-87.6298, 41.8781),
    (-87.9065, 43.0389), (-93.2650, 44.9778), (-100.3510, 44.3683), (-100.7837, 46.8083),
    (-112.0391, 46.5891), (-122.3321, 47.6062), (-122.6765, 45.5231), (-116.2023, 43.6150),
    (-111.8910, 40.7608), (-119.7674, 39.1638), (-118.2437, 34.0522), (-112.0740, 33.4484),
    (-105.9378, 35.6870), (-104.9903, 39.7392), (-104.8202, 41.1400), (-95.9345, 41.2565),
    (-93.6250, 41.5868), (-94.5786, 39.0997), (-95.6890, 39.0473), (-97.5164, 35.4676),
    (-96.7970, 32.7767), (-92.2896, 34.7465), (-90.0490, 35.1495), (-90.1848, 32.2988),
    (-90.0715, 29.9511), (-86.8025, 33.5207), (-84.3880, 33.7490), (-81.6557, 30.3322),
    (-81.0348, 34.0007), (-78.6382, 35.7796), (-77.4360, 37.5407), (-77.0369, 38.9072),
    (-71.0589, 42.3601), (-70.2568, 43.6591),
])


def albers(longitudes, latitudes):
    standard_parallel_1 = np.radians(29.5)
    standard_parallel_2 = np.radians(45.5)
    latitude_origin = np.radians(23.0)
    longitude_origin = np.radians(-96.0)
    longitude = np.radians(np.asarray(longitudes, dtype=float))
    latitude = np.radians(np.asarray(latitudes, dtype=float))
    cone = 0.5 * (np.sin(standard_parallel_1) + np.sin(standard_parallel_2))
    constant = np.cos(standard_parallel_1) ** 2 + 2 * cone * np.sin(standard_parallel_1)
    radius = np.sqrt(constant - 2 * cone * np.sin(latitude)) / cone
    radius_origin = np.sqrt(constant - 2 * cone * np.sin(latitude_origin)) / cone
    angle = cone * (longitude - longitude_origin)
    return radius * np.sin(angle), radius_origin - radius * np.cos(angle)


figure, axis = plt.subplots(figsize=(18, 11))
axis.set_facecolor("#dce8eb")
continental_states = {code: state for code, state in US_STATES.items() if code not in {"AK", "HI"}}
projected_boundaries = []

for state in continental_states.values():
    state_x, state_y = albers(state["lons"], state["lats"])
    projected_boundaries.append((state_x, state_y))
    axis.fill(state_x, state_y, facecolor="#f7f2df", edgecolor="#778482", linewidth=0.8, zorder=0)

city_x, city_y = albers(CITY_LON_LAT[:, 0], CITY_LON_LAT[:, 1])
for start, end in zip(PUBLISHED_TOUR, PUBLISHED_TOUR[1:]):
    axis.add_patch(
        FancyArrowPatch(
            (city_x[start], city_y[start]),
            (city_x[end], city_y[end]),
            arrowstyle="-|>",
            mutation_scale=11,
            linewidth=1.8,
            color="#982f2f",
            alpha=0.9,
            shrinkA=8,
            shrinkB=8,
            zorder=2,
        )
    )

axis.scatter(city_x, city_y, s=260, color="#fffdf4", edgecolor="#17252a", linewidth=1.5, zorder=3)
for city, (x_coordinate, y_coordinate) in enumerate(zip(city_x, city_y), start=1):
    axis.text(
        x_coordinate,
        y_coordinate,
        str(city),
        ha="center",
        va="center",
        fontsize=9.5,
        weight="bold",
        color="#10191c",
        zorder=4,
    )

axis.annotate(
    "40 → A → B → C → D → E → F → G → 41\nWashington to Boston insertions",
    xy=(city_x[39], city_y[39]),
    xycoords="data",
    xytext=(0.025, 0.045),
    textcoords="axes fraction",
    arrowprops={"arrowstyle": "->", "color": "#274d50", "linewidth": 1.6},
    fontsize=11,
    color="#1d383a",
    bbox={"boxstyle": "square,pad=0.45", "facecolor": "#fffdf4", "edgecolor": "#355c5d"},
    zorder=5,
)

boundary_x = np.concatenate([x[np.isfinite(x)] for x, _ in projected_boundaries])
boundary_y = np.concatenate([y[np.isfinite(y)] for _, y in projected_boundaries])
x_padding = 0.025 * (boundary_x.max() - boundary_x.min())
y_padding = 0.035 * (boundary_y.max() - boundary_y.min())
axis.set_xlim(boundary_x.min() - x_padding, boundary_x.max() + x_padding)
axis.set_ylim(boundary_y.min() - y_padding, boundary_y.max() + y_padding)
axis.set_title("Dantzig–Fulkerson–Johnson Optimal Tour (1954)", fontsize=21, pad=28)
axis.text(
    0.5,
    1.01,
    "42-city core: 699 adjusted units • 49-city route: 12,345 reported road miles",
    transform=axis.transAxes,
    ha="center",
    fontsize=12,
)
axis.set_aspect("equal")
axis.axis("off")
figure.tight_layout()

svg_buffer = StringIO()
figure.savefig(svg_buffer, format="svg", bbox_inches="tight")
plt.close(figure)
svg_text = svg_buffer.getvalue().replace(
    "<svg ",
    '<svg style="display:block; width:100%; height:auto;" ',
    1,
)
display(HTML(f'<div style="width:100%; max-width:100%; overflow:hidden;">{svg_text}</div>'))

## Comparison with the 1954 Paper

| Method | Result | Interpretation |
|---|---:|---|
| Paper's 42-city cutting-plane proof | 699 adjusted units | Published optimum; 12,345 miles before scaling and rounding |
| Binary loop-cut reconstruction | 699 adjusted units | Three master solves and nine generated subtour cuts |
| PyOPL MTZ model solved by Gurobi | 699 adjusted units | Independent exact mixed-integer optimum with zero gap |

The methods agree on every undirected tour edge. Gurobi returns the cycle in reverse, which is equivalent because the distance matrix is symmetric. Inserting Baltimore, Wilmington, Philadelphia, Newark, New York, Hartford, and Providence between Washington and Boston produces the paper's complete 49-city route.

## References

- G. B. Dantzig, R. Fulkerson, and S. Johnson, "Solution of a Large-Scale Traveling-Salesman Problem," *Journal of the Operations Research Society of America*, 2(4), 393–410, 1954. [https://www.jstor.org/stable/166695](https://www.jstor.org/stable/166695)
- G. Reinelt, *TSPLIB—A Traveling Salesman Problem Library*, ORSA Journal on Computing, 3(4), 376–384, 1991. [https://doi.org/10.1287/ijoc.3.4.376](https://doi.org/10.1287/ijoc.3.4.376) The `dantzig42` instance supplies the machine-readable matrix and display coordinates used here.
- C. E. Miller, A. W. Tucker, and R. A. Zemlin. 1960. Integer Programming Formulation of Traveling Salesman Problems. J. ACM 7, 4 (Oct. 1960), 326–329. [https://doi.org/10.1145/321043.321046](https://doi.org/10.1145/321043.321046)